In [1]:
import re
import json
import pickle
import numpy as np
import pandas as pd

from pathlib import Path
from scipy import sparse

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, hamming_loss, classification_report

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PATH = PROJECT_ROOT / "dataset" / "processed" / "attack_dataset_stage1_frequent.csv"

df = pd.read_csv(DATA_PATH)

df["Cleaned_Text"] = df["Cleaned_Text"].astype(str)
df["Labels"] = df["Labels"].astype(str)

print("Shape:", df.shape)
display(df.head())

Shape: (20840, 3)


,Cleaned_Text,Tokenized_Text,Labels
0,$BASE64 encoded payload,base64 encoded payload,T1027
1,$INSTDIR File $INSTDIR\o15bmldpqdxcin.dll File...,instdir file instdir o15bmldpqdxcin.dll file i...,T1027
2,$LogFile Timeline for File Activity DFIR File ...,logfile timeline for file activity dfir file s...,T1070
3,$url = URL_TOKEN This line sets the value of t...,url url_token this line sets the value of the ...,T1105
4,%%i in ( WINDOWS_PATH programdata list.txt) do...,i in windows_path programdata list.txt do net ...,T1070


In [3]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.preprocessing import MultiLabelBinarizer

def split_labels(label_str):
    return [
        label.strip()
        for label in str(label_str).split(",")
        if label.strip()
    ]

df["label_list"] = df["Labels"].apply(split_labels)

all_labels = sorted({
    label
    for labels in df["label_list"]
    for label in labels
})

mlb = MultiLabelBinarizer(classes=all_labels)
Y_all = mlb.fit_transform(df["label_list"])

# Iterative multilabel stratification: train 70%, temp 30%
split_train_temp = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=RANDOM_STATE
)
train_idx, temp_idx = next(
    split_train_temp.split(np.zeros(len(df)), Y_all)
)

# Chia temp thành validation 15% và test 15%
split_val_test = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=RANDOM_STATE
)
val_relative_idx, test_relative_idx = next(
    split_val_test.split(np.zeros(len(temp_idx)), Y_all[temp_idx])
)
val_idx = temp_idx[val_relative_idx]
test_idx = temp_idx[test_relative_idx]

train_df = df.iloc[train_idx].copy()
val_df = df.iloc[val_idx].copy()
test_df = df.iloc[test_idx].copy()

X_train = train_df["Cleaned_Text"].values
X_val = val_df["Cleaned_Text"].values
X_test = test_df["Cleaned_Text"].values
Y_train = Y_all[train_idx]
Y_val = Y_all[val_idx]
Y_test = Y_all[test_idx]

label_classes = list(mlb.classes_)

print("Train:", len(X_train))
print("Val:", len(X_val))
print("Test:", len(X_test))
print("Number of labels:", len(label_classes))
print("Y_train:", Y_train.shape)
print("Y_val:", Y_val.shape)
print("Y_test:", Y_test.shape)

def split_diagnostics(name, indices):
    label_counts = Y_all[indices].sum(axis=0)
    return {
        "split": name,
        "samples": len(indices),
        "absent_labels": int((label_counts == 0).sum()),
        "minimum_label_support": int(label_counts.min()),
        "labels_with_less_than_3_samples": int((label_counts < 3).sum()),
        "average_labels_per_sample": float(Y_all[indices].sum(axis=1).mean()),
    }

display(pd.DataFrame([
    split_diagnostics("train", train_idx),
    split_diagnostics("validation", val_idx),
    split_diagnostics("test", test_idx),
]))

assert not (set(train_idx) & set(val_idx))
assert not (set(train_idx) & set(test_idx))
assert not (set(val_idx) & set(test_idx))

Train: 14600
Val: 3130
Test: 3110
Number of labels: 108
Y_train: (14600, 108)
Y_val: (3130, 108)
Y_test: (3110, 108)


,split,samples,absent_labels,minimum_label_support,labels_with_less_than_3_samples,average_labels_per_sample
0,train,14600,0,21,0,1.094247
1,validation,3130,0,4,0,1.094569
2,test,3110,0,4,0,1.100643


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
import re

def cti_tokenizer(text):
    text = str(text).lower()

    tokens = re.findall(
        r"[a-z0-9_]+(?:[./:-][a-z0-9_]+)*",
        text
    )

    return tokens


tfidf_word = TfidfVectorizer(
    tokenizer=cti_tokenizer,
    token_pattern=None,
    lowercase=False,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    max_features=80000,
    sublinear_tf=True
)

tfidf_char = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_df=0.95,
    max_features=80000,
    sublinear_tf=True
)

tfidf_vectorizer = FeatureUnion([
    ("word_tfidf", tfidf_word),
    ("char_tfidf", tfidf_char)
])

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_val_tfidf = tfidf_vectorizer.transform(X_val)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

baseline_model = OneVsRestClassifier(
    LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        solver="liblinear",
        random_state=RANDOM_STATE
    )
)

baseline_model.fit(X_train_tfidf, Y_train)

Y_score_val = baseline_model.predict_proba(X_val_tfidf)
Y_score_test = baseline_model.predict_proba(X_test_tfidf)

print("Model trained.")
print("Y_score_val:", Y_score_val.shape)
print("Y_score_test:", Y_score_test.shape)

Model trained.
Y_score_val: (3130, 108)
Y_score_test: (3110, 108)


In [5]:

Y_score_val = baseline_model.predict_proba(X_val_tfidf)
Y_score_test = baseline_model.predict_proba(X_test_tfidf)

print("Y_score_val shape:", Y_score_val.shape)
print("Y_score_test shape:", Y_score_test.shape)

print("Min score:", Y_score_test.min())
print("Max score:", Y_score_test.max())

Y_score_val shape: (3130, 108)
Y_score_test shape: (3110, 108)
Min score: 0.0007281743059955827
Max score: 0.9999994302883993


In [6]:
def predict_with_threshold_fallback(
    y_score,
    threshold=0.5,
    min_k=1,
    max_k=3
):
    y_pred = np.zeros_like(y_score, dtype=int)

    for i, score_row in enumerate(y_score):
        selected = np.where(score_row >= threshold)[0]

        # Nếu không có nhãn nào vượt threshold, lấy top-1
        if len(selected) < min_k:
            selected = np.argsort(score_row)[-min_k:]

        # Nếu có quá nhiều nhãn vượt threshold, giữ max_k nhãn cao nhất
        if len(selected) > max_k:
            selected = selected[np.argsort(score_row[selected])[-max_k:]]

        y_pred[i, selected] = 1

    return y_pred

In [7]:
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    hamming_loss,
    accuracy_score
)

def evaluate_multilabel(y_true, y_pred, model_name="model"):
    avg_true_labels = y_true.sum(axis=1).mean()
    avg_pred_labels = y_pred.sum(axis=1).mean()
    zero_pred_samples = int((y_pred.sum(axis=1) == 0).sum())

    result = {
        "model": model_name,
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),

        "micro_precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),

        "micro_recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),

        "hamming_loss": hamming_loss(y_true, y_pred),
        "subset_accuracy": accuracy_score(y_true, y_pred),

        "avg_true_labels": avg_true_labels,
        "avg_pred_labels": avg_pred_labels,
        "zero_pred_samples": zero_pred_samples,
        "zero_pred_percentage": zero_pred_samples / len(y_true) * 100
    }

    return result

In [8]:
def tune_threshold_on_validation(
    y_true_val,
    y_score_val,
    thresholds=None,
    optimize_metric="micro_f1",
    min_k=1,
    max_k=3
):
    if thresholds is None:
        thresholds = np.arange(0.05, 0.96, 0.05)

    rows = []

    for threshold in thresholds:
        y_pred_val = predict_with_threshold_fallback(
            y_score_val,
            threshold=threshold,
            min_k=min_k,
            max_k=max_k
        )

        result = evaluate_multilabel(
            y_true_val,
            y_pred_val,
            model_name=f"threshold_{threshold:.2f}"
        )

        result["threshold"] = threshold
        rows.append(result)

    results_df = pd.DataFrame(rows)
    results_df = results_df.sort_values(optimize_metric, ascending=False).reset_index(drop=True)

    return results_df

In [9]:
threshold_results_val = tune_threshold_on_validation(
    Y_val,
    Y_score_val,
    thresholds=np.arange(0.05, 0.96, 0.05),
    optimize_metric="micro_f1",
    min_k=1,
    max_k=3
)

display(threshold_results_val)

best_threshold = float(threshold_results_val.iloc[0]["threshold"])

print("Best threshold:", best_threshold)

,model,micro_f1,macro_f1,weighted_f1,micro_precision,macro_precision,weighted_precision,micro_recall,macro_recall,weighted_recall,hamming_loss,subset_accuracy,avg_true_labels,avg_pred_labels,zero_pred_samples,zero_pred_percentage,threshold
0,threshold_0.90,0.694328,0.595072,0.687157,0.710446,0.654125,0.710502,0.678926,0.575829,0.678926,0.006058,0.637700,1.094569,1.046006,0,0.0,0.90
1,threshold_0.95,0.694180,0.596202,0.686494,0.719975,0.666969,0.719336,0.670169,0.566706,0.670169,0.005984,0.646326,1.094569,1.018850,0,0.0,0.95
2,threshold_0.85,0.692874,0.593382,0.685693,0.699049,0.640783,0.698913,0.686807,0.584313,0.686807,0.006171,0.623323,1.094569,1.075399,0,0.0,0.85
3,threshold_0.80,0.692552,0.596776,0.686705,0.685037,0.623564,0.685337,0.700234,0.600086,0.700234,0.006301,0.607348,1.094569,1.118850,0,0.0,0.80
4,threshold_0.75,0.689528,0.596226,0.684705,0.670063,0.615149,0.672845,0.710158,0.608981,0.710158,0.006481,0.591693,1.094569,1.160064,0,0.0,0.75
5,threshold_0.70,0.685295,0.594238,0.681906,0.651328,0.596341,0.656420,0.723001,0.623361,0.723001,0.006730,0.570288,1.094569,1.215016,0,0.0,0.70
6,threshold_0.65,0.676700,0.584460,0.674515,0.626992,0.569204,0.633906,0.734968,0.632971,0.734968,0.007118,0.541853,1.094569,1.283067,0,0.0,0.65
7,threshold_0.60,0.668577,0.579020,0.668483,0.603765,0.550644,0.613645,0.748978,0.644517,0.748978,0.007526,0.511821,1.094569,1.357827,0,0.0,0.60
8,threshold_0.55,0.658910,0.574994,0.660580,0.581343,0.537932,0.594360,0.760362,0.656376,0.760362,0.007978,0.479872,1.094569,1.431629,0,0.0,0.55
9,threshold_0.50,0.644580,0.565976,0.648240,0.553094,0.509418,0.567352,0.772329,0.669596,0.772329,0.008632,0.440256,1.094569,1.528435,0,0.0,0.50


Best threshold: 0.9000000000000001


In [10]:
Y_pred_test_threshold = predict_with_threshold_fallback(
    Y_score_test,
    threshold=best_threshold,
    min_k=1,
    max_k=3
)

print("Y_pred_test_threshold shape:", Y_pred_test_threshold.shape)
print("Average predicted labels:", Y_pred_test_threshold.sum(axis=1).mean())
print("Min predicted labels:", Y_pred_test_threshold.sum(axis=1).min())
print("Max predicted labels:", Y_pred_test_threshold.sum(axis=1).max())

Y_pred_test_threshold shape: (3110, 108)
Average predicted labels: 1.0434083601286173
Min predicted labels: 1
Max predicted labels: 3


In [11]:
test_metrics = evaluate_multilabel(
    Y_test,
    Y_pred_test_threshold,
    model_name=f"TF-IDF word+char + Logistic Regression threshold={best_threshold:.2f}"
)

test_metrics_df = pd.DataFrame([test_metrics])

display(test_metrics_df)

,model,micro_f1,macro_f1,weighted_f1,micro_precision,macro_precision,weighted_precision,micro_recall,macro_recall,weighted_recall,hamming_loss,subset_accuracy,avg_true_labels,avg_pred_labels,zero_pred_samples,zero_pred_percentage
0,TF-IDF word+char + Logistic Regression thresho...,0.692561,0.590539,0.686334,0.711556,0.659613,0.717647,0.674554,0.566894,0.674554,0.006103,0.637621,1.100643,1.043408,0,0.0


In [12]:
def top_k_predictions(y_score, k=3):
    top_k_indices = np.argsort(y_score, axis=1)[:, -k:]

    y_pred = np.zeros_like(y_score, dtype=int)

    for i, indices in enumerate(top_k_indices):
        y_pred[i, indices] = 1

    return y_pred


Y_pred_test_top3 = top_k_predictions(Y_score_test, k=3)

top3_metrics = evaluate_multilabel(
    Y_test,
    Y_pred_test_top3,
    model_name="TF-IDF word+char + Logistic Regression fixed Top-3"
)

comparison_df = pd.DataFrame([
    top3_metrics,
    test_metrics
])

display(comparison_df)

,model,micro_f1,macro_f1,weighted_f1,micro_precision,macro_precision,weighted_precision,micro_recall,macro_recall,weighted_recall,hamming_loss,subset_accuracy,avg_true_labels,avg_pred_labels,zero_pred_samples,zero_pred_percentage
0,TF-IDF word+char + Logistic Regression fixed T...,0.457461,0.409561,0.473359,0.312647,0.288702,0.332596,0.852176,0.745719,0.852176,0.020600,0.004823,1.100643,3.000000,0,0.0
1,TF-IDF word+char + Logistic Regression thresho...,0.692561,0.590539,0.686334,0.711556,0.659613,0.717647,0.674554,0.566894,0.674554,0.006103,0.637621,1.100643,1.043408,0,0.0


### Đánh giá cấu hình đề xuất SOC với threshold cố định = 0.35

Cell này giữ tối thiểu 1 và tối đa 3 nhãn cho mỗi mẫu. `Precision@3` và `Recall@3` được tính từ ba label có xác suất cao nhất, độc lập với threshold, để đo chất lượng ranking phục vụ analyst.

In [13]:
SOC_THRESHOLD = 0.35
SOC_TOP_K = 3

Y_pred_test_soc = predict_with_threshold_fallback(
    Y_score_test,
    threshold=SOC_THRESHOLD,
    min_k=1,
    max_k=SOC_TOP_K
)

def ranking_precision_recall_at_k(y_true, y_score, k=3):
    top_k_indices = np.argpartition(y_score, -k, axis=1)[:, -k:]
    top_k_pred = np.zeros_like(y_true, dtype=int)
    top_k_pred[np.arange(len(y_true))[:, None], top_k_indices] = 1

    hits = (top_k_pred * y_true).sum(axis=1)
    precision_at_k = np.mean(hits / k)
    recall_at_k = np.mean(
        hits / np.maximum(y_true.sum(axis=1), 1)
    )
    return float(precision_at_k), float(recall_at_k)

precision_at_3, recall_at_3 = ranking_precision_recall_at_k(
    Y_test, Y_score_test, k=SOC_TOP_K
)

soc_metrics = evaluate_multilabel(
    Y_test,
    Y_pred_test_soc,
    model_name=(
        f"TF-IDF word+char + OvR Logistic Regression "
        f"SOC threshold={SOC_THRESHOLD:.2f}"
    )
)
soc_metrics["precision_at_3"] = precision_at_3
soc_metrics["recall_at_3"] = recall_at_3

soc_metrics_df = pd.DataFrame([soc_metrics])
preferred_columns = [
    "model",
    "micro_f1", "macro_f1", "weighted_f1",
    "micro_precision", "macro_precision", "weighted_precision",
    "micro_recall", "macro_recall", "weighted_recall",
    "precision_at_3", "recall_at_3",
    "hamming_loss", "subset_accuracy",
    "avg_true_labels", "avg_pred_labels",
]
display(soc_metrics_df[preferred_columns])

print(f"SOC threshold: {SOC_THRESHOLD:.2f}")
print(f"Precision@3: {precision_at_3:.4f}")
print(f"Recall@3: {recall_at_3:.4f}")
print(f"Average predicted labels: {Y_pred_test_soc.sum(axis=1).mean():.4f}")
print(
    "Predicted-label range:",
    int(Y_pred_test_soc.sum(axis=1).min()),
    "to",
    int(Y_pred_test_soc.sum(axis=1).max()),
)

,model,micro_f1,macro_f1,weighted_f1,micro_precision,macro_precision,weighted_precision,micro_recall,macro_recall,weighted_recall,precision_at_3,recall_at_3,hamming_loss,subset_accuracy,avg_true_labels,avg_pred_labels
0,TF-IDF word+char + OvR Logistic Regression SOC...,0.586077,0.510261,0.599505,0.457073,0.409177,0.480865,0.816535,0.711131,0.816535,0.312647,0.857395,0.011754,0.291961,1.100643,1.966238


SOC threshold: 0.35
Precision@3: 0.3126
Recall@3: 0.8574
Average predicted labels: 1.9662
Predicted-label range: 1 to 3


### Các nhãn có hiệu quả dự đoán dưới 30%

Mặc định cell lọc theo per-label F1 `< 0.30` trên test set với cấu hình SOC threshold 0.35. Có thể đổi `LOW_PERFORMANCE_METRIC` thành `precision` hoặc `recall` nếu muốn phân tích theo metric khác.

In [14]:
from sklearn.metrics import precision_recall_fscore_support

LOW_PERFORMANCE_THRESHOLD = 0.30
LOW_PERFORMANCE_METRIC = "f1"  # chọn: "precision", "recall" hoặc "f1"

label_precision, label_recall, label_f1, label_support = (
    precision_recall_fscore_support(
        Y_test,
        Y_pred_test_soc,
        average=None,
        zero_division=0
    )
)

per_label_soc_report = pd.DataFrame({
    "label": label_classes,
    "true_sample_count": label_support.astype(int),
    "predicted_sample_count": Y_pred_test_soc.sum(axis=0).astype(int),
    "precision": label_precision,
    "recall": label_recall,
    "f1": label_f1,
})

if LOW_PERFORMANCE_METRIC not in {"precision", "recall", "f1"}:
    raise ValueError("LOW_PERFORMANCE_METRIC phải là precision, recall hoặc f1")

low_performance_labels = (
    per_label_soc_report[
        per_label_soc_report[LOW_PERFORMANCE_METRIC] < LOW_PERFORMANCE_THRESHOLD
    ]
    .sort_values(
        [LOW_PERFORMANCE_METRIC, "true_sample_count"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

print(
    f"Labels with {LOW_PERFORMANCE_METRIC} < "
    f"{LOW_PERFORMANCE_THRESHOLD:.0%}: {len(low_performance_labels)} / "
    f"{len(label_classes)}"
)
print(
    "Total true test samples/occurrences of these labels:",
    int(low_performance_labels["true_sample_count"].sum())
)
display(low_performance_labels)

Labels with f1 < 30%: 14 / 108
Total true test samples/occurrences of these labels: 100


,label,true_sample_count,predicted_sample_count,precision,recall,f1
0,T1211,5,3,0.000000,0.000000,0.000000
1,T1587,5,1,0.000000,0.000000,0.000000
2,T1586,9,16,0.062500,0.111111,0.080000
3,T1421,4,15,0.066667,0.250000,0.105263
4,T1555,11,17,0.117647,0.181818,0.142857
5,T1554,6,22,0.090909,0.333333,0.142857
6,T1631,5,9,0.111111,0.200000,0.142857
7,T1020,6,7,0.142857,0.166667,0.153846
8,T1001,6,18,0.111111,0.333333,0.166667
9,T1584,6,17,0.117647,0.333333,0.173913
